In [ ]:
# ============================================================
# Project root & path handling
# ------------------------------------------------------------
# Default: working directory
# ============================================================

from pathlib import Path
import os
import sys
import json
import numpy as np
import pandas as pd

# Determine project root
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "../..")).expanduser().resolve()

# Make the project's `src` package importable (e.g. src.lib.assay_priority)
sys.path.insert(0, str(PROJECT_ROOT))

# Convenience function for building repo-relative paths
def p(rel_path):
    """
    Build an absolute path from a path relative to the project root.
    """
    return PROJECT_ROOT / rel_path

print("PROJECT_ROOT set to:", PROJECT_ROOT)

In [ ]:
# ============================================================
# Create temporary and output directories 
# ============================================================

from src.lib.data_paths import get_data_paths

DATA_DIR = p("data")
_paths = get_data_paths(DATA_DIR)
_paths.make_output_dirs()

INPUT_MAVES_DIR = _paths.input_maves_dir
MAVE_DATA_DIR = _paths.mave_data_dir
SUPPLEMENTARY_DATA_DIR = _paths.supplementary_data_dir
MAVE_CALIBRATION_DIR = _paths.mave_calibration_dir
MAVE_CALIBRATION_ODDSPATH_DIR = _paths.mave_calibration_oddspath_dir
PREDICTOR_CALIBRATION_GENE_SPECIFIC_DIR = _paths.predictor_calibration_gene_specific_dir
RECLASSIFICATION_DIR = _paths.reclassification_dir

In [ ]:
import pandas as pd
pp = pp = pd.read_csv(MAVE_DATA_DIR/"integrated_variant_effect_dataset.tsv.gz", sep = '\t')

In [ ]:
excalibr = pd.read_excel(SUPPLEMENTARY_DATA_DIR/"Supplementary_Data_4.xlsx", sheet_name = 'ExCALIBR_calibrations')
REVEL_gs = pd.read_excel(SUPPLEMENTARY_DATA_DIR/"Supplementary_Data_4.xlsx", sheet_name = 'REVEL_gene_specific_calibration')
AM_gs = pd.read_excel(SUPPLEMENTARY_DATA_DIR/"Supplementary_Data_4.xlsx", sheet_name = 'AM_gene_specific_calibrations')
MP2_gs = pd.read_excel(SUPPLEMENTARY_DATA_DIR/"Supplementary_Data_4.xlsx", sheet_name = 'MP2_gene_specific_calibrations')
OP = pd.read_excel(SUPPLEMENTARY_DATA_DIR/"Supplementary_Data_4.xlsx", sheet_name = 'OddsPath_calibrations')

In [ ]:
import pandas as pd

# macOS decomposes non-ASCII filenames to NFD on extraction (e.g. RAD51C_Olvera-León's
# accented o), which silently breaks the exact-string merge below against pp['Dataset']
# (typically NFC) with no error -- the affected dataset just gets zero functional points.
# load_excalibr_calibrations.py normalizes at the source now, but normalize again here
# too in case this sheet was written by an older run or edited by hand.
import unicodedata
excalibr['dataset'] = excalibr['dataset'].apply(lambda s: unicodedata.normalize('NFC', s) if isinstance(s, str) else s)

excalibr['base'] = excalibr['dataset'].str.replace('_clinvar_2018', '', regex=False)

#Split main vs 2018 rows
excalibr_main = excalibr[~excalibr['dataset'].str.endswith('_clinvar_2018')].copy()
excalibr_2018 = excalibr[excalibr['dataset'].str.endswith('_clinvar_2018')].copy()

excalibr_2018['base'] = excalibr_2018['dataset'].str.replace('_clinvar_2018', '', regex=False)
excalibr_2018 = excalibr_2018.add_suffix('_2018')
excalibr_main = excalibr_main.add_suffix('_2025')


#Merge. Outer join: some calibrations (e.g. BRCA1_Findlay_2018) exist only as
#a _clinvar_2018 file with no non-2018 counterpart. A left join from
#excalibr_main would silently drop those calibrations entirely.
excalibr_merged = excalibr_main.merge(
    excalibr_2018,
    left_on='base_2025',
    right_on='base_2018',
    how='outer'
)

#Unify the base name so datasets with only a 2018 or only a 2025 calibration
#can still be joined onto `pp` below
excalibr_merged['base'] = excalibr_merged['base_2025'].fillna(excalibr_merged['base_2018'])
excalibr_merged = excalibr_merged.drop(columns=['base_2025', 'base_2018'], errors='ignore')

In [ ]:
pp['Dataset'] = pp['Dataset'].apply(lambda s: unicodedata.normalize('NFC', s) if isinstance(s, str) else s)
pp_ex = pd.merge(pp, excalibr_merged, left_on = 'Dataset', right_on = 'base', how = 'left')

In [ ]:
import numpy as np
import pandas as pd
import re

#function to get points from the ExCALIBR intervals 

def get_points_from_intervals(row, suffix=""):
    score = row["auth_reported_score"]

    for col in row.index:
        if col.startswith("range_") and col.endswith(suffix):
            cell = row[col]
            if cell is None or pd.isna(cell):
                continue

            if isinstance(cell, str):
                # A cell can hold more than one disjoint range (e.g.
                # DDX3X_Radford_2023's +1/-1 points), comma-separated.
                ranges = []
                for chunk in cell.split(","):
                    lo, hi = chunk.strip().split()
                    lower = -np.inf if lo.lower() == "-inf" else float(lo)
                    upper =  np.inf if hi.lower() == "inf"  else float(hi)
                    ranges.append((lower, upper))
            else:
                ranges = [cell]  # works for tuple/list/Interval

            for lower, upper in ranges:
                in_lower = (lower == -np.inf) or (score >= lower)
                in_upper = (upper == np.inf) or (score < upper)

                if in_lower and in_upper:
                    m = re.search(r"^range_(-?\d+)", col)
                    return int(m.group(1)) if m else None

    return None


In [ ]:
import re

#convert author reported score to numeric 
pp_ex["auth_reported_score"] = pd.to_numeric(
    pp_ex["auth_reported_score"].astype(str).str.strip(),
    errors="coerce"
)

#get points for calibrations that have been performed on clinvar_2025 controls
pp_ex["ExC_points_2025"] = pp_ex.apply(
    lambda row: get_points_from_intervals(row, suffix="_2025"),
    axis=1
)

#get points for calibrations that have been performed on clinvar_2018 controls
pp_ex["ExC_points_2018"] = pp_ex.apply(
    lambda row: get_points_from_intervals(row, suffix="_2018"),
    axis=1
)

In [ ]:
pp_ex_OP = pd.merge(pp_ex, OP, on = 'Dataset', how = 'left')

In [ ]:
#annotate standardized functional classification

def annotate_func_class(row):
    # Formerly, this translated each dataset's author-reported functional
    # class label (row['auth_reported_func_class']) into a standardized class
    # using a per-dataset dictionary, now preserved for documentation/historical
    # purposes only in src/lib/func_class.py (FUNC_CLASS_LABEL_MAP).
    #
    # New: use the category MaveDB itself assigns, so our standardized class
    # stays consistent with MaveDB's categorization.
    category = row['auth_reported_func_class_category']
    if pd.isna(category):
        return None

    category_map = {
        'normal': 'Normal',
        'abnormal': 'Abnormal',
        'not_specified': 'Indeterminate'
    }
    return category_map.get(str(category).strip().lower(), None)

pp_ex_OP['StandardizedClass'] = pp_ex_OP.apply(annotate_func_class, axis=1)

In [ ]:
mave = pd.read_excel(INPUT_MAVES_DIR / "Supplementary_Data_3.xlsx", sheet_name="Curation", header=0)

In [ ]:
pp_ex_OP = pd.merge(pp_ex_OP, mave[['Dataset Name','Score Intervals Reported?','Functional Classification Provided?']],
                   left_on = 'Dataset', right_on = 'Dataset Name', 
                   how = 'left')

In [ ]:
def parse_bound(bound_str):
    """
    Parse a string boundary from an interval definition into a numeric value.

    Handles special cases for infinity (e.g., '-inf', 'inf') and converts
    numeric strings to floats.
    """
    
    bound_str = bound_str.strip().lower()
    if bound_str in ['-inf', '-infinity']:
        return float('-inf')
    elif bound_str in ['inf', 'infinity', '+inf']:
        return float('inf')
    else:
        return float(bound_str)

def score_in_interval(score, interval):
    """
    Determine whether a numeric score falls within a specified interval.

    Intervals are defined as strings using set notation, e.g.:
        '(-inf, 0]', '(0, 0.4)', '[0.4, inf)'

    Inclusivity/exclusivity of bounds is inferred from brackets:
        '[' or ']'  -> inclusive
        '(' or ')'  -> exclusive
    """
    try:
        left = interval[0]
        right = interval[-1]
        lower_str, upper_str = interval[1:-1].split(',')

        lower = parse_bound(lower_str)
        upper = parse_bound(upper_str)

        lower_check = score >= lower if left == '[' else score > lower
        upper_check = score <= upper if right == ']' else score < upper

        return lower_check and upper_check
    except:
        return False

def find_matching_class(row, score):

    """
    Identify the functional class corresponding to a score for a given row.

    Iterates over interval definitions (Interval 1–9) and returns the first
    Class whose interval contains the score.
    """
    for i in range(1, 10): 
        interval_col = f'Interval {i} Range'
        class_col = f'Interval {i} Class'

        if interval_col in row and pd.notna(row[interval_col]):
            interval = row[interval_col]
            if score_in_interval(score, interval):
                return row[class_col]
    return 'Not specified'

In [ ]:
mask_OP = (
    (pp_ex_OP['Functional Classification Provided?'].isin(['No']) | pp_ex_OP['Functional Classification Provided?'].isna()) &
    (pp_ex_OP['Score Intervals Reported?'] == 'Reported')
)

pp_ex_OP.loc[mask_OP, 'StandardizedClass'] = pp_ex_OP.loc[mask_OP].apply(
    lambda row: find_matching_class(row, row['auth_reported_score']),
    axis=1
)

In [ ]:
pp_ex_OP['StandardizedClass'] = pp_ex_OP['StandardizedClass'].str.upper()

In [ ]:
import numpy as np

#annotate points for OddsPath

def annotate_OP_points(
    row,
    value_col_normal,
    value_col_abnormal,
    class_col="StandardizedClass"
):
    cls = row[class_col]

    # Missing class
    if pd.isna(cls):
        return np.nan

    # choose correct column
    if cls == "NORMAL":
        raw_value = row[value_col_normal]
    elif cls == "ABNORMAL":
        raw_value = row[value_col_abnormal]
    else:
        return np.nan

    # try converting to float
    try:
        value = float(raw_value)
    except (TypeError, ValueError):
        # If conversion fails (string like "Cannot calculate"), return NaN
        return np.nan

    # --- NORMAL branch (BS3 scale) ---
    if cls == "NORMAL":
        if value < 0.053:
            return -4
        elif value < 0.23:
            return -2
        elif value < 0.48:
            return -1
        else:
            return 0

    # --- ABNORMAL branch (PS3 scale) ---
    elif cls == "ABNORMAL":
        if value > 350:
            return 8
        elif value > 18.7:
            return 4
        elif value > 4.3:
            return 2
        elif value > 2.1:
            return 1
        else:
            return 0

    return np.nan


In [ ]:
pp_ex_OP["OP_points"] = pp_ex_OP.apply(
    annotate_OP_points,
    value_col_normal="OddsNormal",
    value_col_abnormal="OddsAbnormal",
    axis=1
)

In [ ]:
priority_genes_2018 = ['BRCA1', 'PTEN', 'MSH2'] #TP53 gets OP points (which already considers 2018 clinvar controls)
op_genes = ['F9', 'TP53']

pp_ex_OP['Fxn_points'] = pp_ex_OP['ExC_points_2025']

#2018 calibrations
mask_2018 = pp_ex_OP['Gene'].isin(priority_genes_2018)
pp_ex_OP.loc[mask_2018, 'Fxn_points'] = pd.to_numeric(
    pp_ex_OP.loc[mask_2018, 'ExC_points_2018']
    .fillna(pp_ex_OP.loc[mask_2018, 'ExC_points_2025']),
    errors='coerce'
)

#OP overrides
mask_op = pp_ex_OP['Gene'].isin(op_genes)
pp_ex_OP.loc[mask_op, 'Fxn_points'] = pp_ex_OP.loc[mask_op, 'OP_points']


In [ ]:
#fill na with 0 
pp_ex_OP['Fxn_points'] = pp_ex_OP['Fxn_points'].fillna(0)

In [ ]:
#fill in the evidence strength for variants with a REVEL score given their score in Berquist et al.

# Round to 3 decimal places first -- the bin boundaries below are exact
# three-decimal cutoffs, so an unrounded score can fall in the unassigned
# gap between two adjacent bins instead of the bin it belongs in.
pp_ex_OP['REVEL_rounded'] = pp_ex_OP['REVEL'].round(3)

REVEL_conditions = [
    pp_ex_OP['REVEL_rounded'] <= 0.016, 
    (pp_ex_OP['REVEL_rounded'] >= 0.017) & (pp_ex_OP['REVEL_rounded'] <= 0.052),  
    (pp_ex_OP['REVEL_rounded'] >= 0.053) & (pp_ex_OP['REVEL_rounded'] <= 0.183),  
    (pp_ex_OP['REVEL_rounded'] >= 0.184) & (pp_ex_OP['REVEL_rounded'] <= 0.290),
    (pp_ex_OP['REVEL_rounded'] >= 0.644) & (pp_ex_OP['REVEL_rounded'] <= 0.772),  
    (pp_ex_OP['REVEL_rounded'] >= 0.773) & (pp_ex_OP['REVEL_rounded'] <= 0.878),  
    (pp_ex_OP['REVEL_rounded'] >= 0.879) & (pp_ex_OP['REVEL_rounded'] <= 0.931),  
    pp_ex_OP['REVEL_rounded'] >= 0.932                   
]
REVEL_values = ['BP4_Strong', 
    'BP4_Moderate+', 
    'BP4_Moderate', 
    'BP4_Supporting',
    'PP3_Supporting', 
    'PP3_Moderate', 
    'PP3_Moderate+', 
    'PP3_Strong']

# Apply classifications
pp_ex_OP['REVEL_GenomeWide_Code'] = np.select(REVEL_conditions, REVEL_values, default= pd.NA)

In [ ]:
#fill in the evidence strength for variants with a MutPred2 score given their score in the Pejaver et al paper

# Round to 3 decimal places first -- see the REVEL cell above for why.
pp_ex_OP['MutPred2_rounded'] = pp_ex_OP['MutPred2'].round(3)

MP2_conditions = [
    pp_ex_OP['MutPred2_rounded'] <= 0.010,  
    (pp_ex_OP['MutPred2_rounded'] >= 0.011) & (pp_ex_OP['MutPred2_rounded'] <= 0.031),  
    (pp_ex_OP['MutPred2_rounded'] >= 0.032) & (pp_ex_OP['MutPred2_rounded'] <= 0.197),  
    (pp_ex_OP['MutPred2_rounded'] >= 0.198) & (pp_ex_OP['MutPred2_rounded'] <= 0.391),
    (pp_ex_OP['MutPred2_rounded'] >= 0.737) & (pp_ex_OP['MutPred2_rounded'] <= 0.828), 
    (pp_ex_OP['MutPred2_rounded'] >= 0.829) & (pp_ex_OP['MutPred2_rounded'] <= 0.894),
    (pp_ex_OP['MutPred2_rounded'] >= 0.895) & (pp_ex_OP['MutPred2_rounded'] <= 0.931),  
    pp_ex_OP['MutPred2_rounded'] >= 0.932       
]

MP2_values = ['BP4_Strong',
    'BP4_Moderate+', 
    'BP4_Moderate', 
    'BP4_Supporting',
    'PP3_Supporting', 
    'PP3_Moderate', 
    'PP3_Moderate+', 
    'PP3_Strong']

# Apply classifications
pp_ex_OP['MP2_GenomeWide_Code'] = np.select(MP2_conditions, MP2_values, default= pd.NA)

In [ ]:
#fill in the evidence strength for variants with a Alphamissense score given their score in the Berquist et al paper

# Round to 3 decimal places first -- see the REVEL cell above for why.
pp_ex_OP['AM_score_rounded'] = pp_ex_OP['AM_score'].round(3)

AM_conditions = [
    pp_ex_OP['AM_score_rounded'] <= 0.070,  
    (pp_ex_OP['AM_score_rounded'] >= 0.071) & (pp_ex_OP['AM_score_rounded'] <= 0.099),  
    (pp_ex_OP['AM_score_rounded'] >= 0.100) & (pp_ex_OP['AM_score_rounded'] <= 0.169), 
    (pp_ex_OP['AM_score_rounded'] >= 0.792) & (pp_ex_OP['AM_score_rounded'] <= 0.905),  
    (pp_ex_OP['AM_score_rounded'] >= 0.906) & (pp_ex_OP['AM_score_rounded'] <= 0.971),  
    (pp_ex_OP['AM_score_rounded'] >= 0.972) & (pp_ex_OP['AM_score_rounded'] <= 0.989),  
    pp_ex_OP['AM_score_rounded'] >= 0.990      
]

AM_values = ['BP4_Moderate+', 
    'BP4_Moderate', 
    'BP4_Supporting',
    'PP3_Supporting', 
    'PP3_Moderate', 
    'PP3_Moderate+', 
    'PP3_Strong']

# Apply classifications
pp_ex_OP['AM_GenomeWide_Code'] = np.select(AM_conditions, AM_values, default= pd.NA)

In [ ]:
#create a gene list for all the genes in the dataframe 
pp_gene_list = list(pp['Gene'].unique())

In [ ]:
#filter for genes that are in the dataframe 
REVEL_gene_spc = REVEL_gs[REVEL_gs['Gene'].isin(pp_gene_list)]

In [ ]:
#filter for genes that are in the dataframe 
AM_gene_spc = AM_gs[AM_gs['Gene'].isin(pp_gene_list)]

In [ ]:
#filter for genes that are in the dataframe 
MP2_gene_spc = MP2_gs[MP2_gs['Gene'].isin(pp_gene_list)]

In [ ]:
import pandas as pd
import numpy as np

def classify_variants(variants_df, thresholds_df, class_col, gene_col='Gene', score_col='score'):
    """
    Classify each row's score against its gene's threshold row (vectorized).

    Logic:
    - BP4 thresholds are UPPER bounds (score <= threshold = that category)
      Lower score = stronger benign evidence
    - PP3 thresholds are LOWER bounds (score >= threshold = that category)
      Higher score = stronger pathogenic evidence
    - Between BP4_Supporting and PP3_Supporting = Indeterminate
    """
    display_names = {
        "BP4_Very Strong": "BP4_Very Strong",
        "BP4_Strong": "BP4_Strong",
        "BP4_Moderate+": "BP4_Moderate+",
        "BP4_Moderate": "BP4_Moderate",
        "BP4_Supporting": "BP4_Supporting",
        "PP3_Supporting": "PP3_Supporting",
        "PP3_Moderate": "PP3_Moderate",
        "PP3_Moderate+": "PP3_Moderate+",
        "PP3_Strong": "PP3_Strong",
        "PP3_Very Strong": "PP3_Very Strong"
    }

    # BP4 columns in order from strongest to weakest (thresholds are upper bounds)
    bp4_columns = ['BP4_Very Strong', 'BP4_Strong', 'BP4_Moderate+', 'BP4_Moderate', 'BP4_Supporting']

    # PP3 columns in order from strongest to weakest (thresholds are lower bounds)
    pp3_columns = ['PP3_Very Strong', 'PP3_Strong', 'PP3_Moderate+', 'PP3_Moderate', 'PP3_Supporting']

    result_df = variants_df.copy()
    score = result_df[score_col]

    # Bring each row's gene-specific thresholds alongside it via a left merge on gene_col
    # -- thresholds_df has one row per gene, so this can't duplicate or reorder rows.
    threshold_cols = [c for c in bp4_columns + pp3_columns if c in thresholds_df.columns]
    thresholds = result_df[[gene_col]].merge(
        thresholds_df[[gene_col] + threshold_cols], on=gene_col, how='left'
    )

    # np.select picks the first true condition, matching the strongest-to-weakest loop
    # order below. A NaN score or NaN threshold makes every comparison false, so those
    # rows fall through to the "__NA__" default -- no explicit isna check needed.
    conditions = []
    choices = []
    for col in bp4_columns:
        if col in threshold_cols:
            conditions.append(score.le(thresholds[col].values))
            choices.append(display_names[col])
    for col in pp3_columns:
        if col in threshold_cols:
            conditions.append(score.ge(thresholds[col].values))
            choices.append(display_names[col])

    classified = np.select(conditions, choices, default="__NA__")
    result_df[class_col] = pd.Series(classified, index=result_df.index).mask(
        classified == "__NA__", np.nan
    )

    return result_df

In [ ]:
pp_ex_OP = classify_variants(pp_ex_OP, REVEL_gene_spc, "REVEL_GeneSpecific_Code", gene_col='Gene', score_col='REVEL')
pp_ex_OP = classify_variants(pp_ex_OP, AM_gene_spc, "AM_GeneSpecific_Code", gene_col='Gene', score_col='AM_score')
pp_ex_OP = classify_variants(pp_ex_OP, MP2_gene_spc, "MP2_GeneSpecific_Code", gene_col='Gene', score_col='MutPred2')

In [ ]:
def calculate_p_points(df, col, prefix):

    points_col = {
        "PP3_Supporting": 1,
        "PP3_Moderate": 2,
        "PP3_Moderate+": 3,
        "PP3_Strong": 4,
        "PP3_Very Strong": 8,

        "BP4_Supporting": -1,
        "BP4_Moderate": -2,
        "BP4_Moderate+": -3,
        "BP4_Strong": -4,
        "BP4_Very Strong": -8
    }

    # .map returns NaN for values it can't find in points_col, which already covers
    # both NA scores (str(NaN) == "nan", not a key) and unrecognized codes.
    df[f"Points{prefix}"] = df[col].astype(str).str.strip().map(points_col)

    return df

In [ ]:
#REVEL

pp_ex_OP = calculate_p_points(
    pp_ex_OP,
    col = "REVEL_GenomeWide_Code",
    prefix="_REVEL_GenomeWide"
)

pp_ex_OP = calculate_p_points(
    pp_ex_OP,
    col = "REVEL_GeneSpecific_Code",
    prefix="_REVEL_GeneSpecific"
)


In [ ]:
#AM

pp_ex_OP = calculate_p_points(
    pp_ex_OP,
    col = "AM_GenomeWide_Code",
    prefix="_AM_GenomeWide"
)

pp_ex_OP = calculate_p_points(
    pp_ex_OP,
    col = "AM_GeneSpecific_Code",
    prefix="_AM_GeneSpecific"
)

In [ ]:
#MP2

pp_ex_OP = calculate_p_points(
    pp_ex_OP,
    col = "MP2_GenomeWide_Code",
    prefix="_MP2_GenomeWide"
)

pp_ex_OP = calculate_p_points(
    pp_ex_OP,
    col = "MP2_GeneSpecific_Code",
    prefix="_MP2_GeneSpecific"
)

In [ ]:
pp_ex_OP['Points_REVEL_GeneSpecific_GenomeWide'] = pp_ex_OP['Points_REVEL_GeneSpecific'].fillna(pp_ex_OP['Points_REVEL_GenomeWide']).fillna(0)

pp_ex_OP['Points_AM_GeneSpecific_GenomeWide'] = pp_ex_OP['Points_AM_GeneSpecific'].fillna(pp_ex_OP['Points_AM_GenomeWide']).fillna(0)

pp_ex_OP['Points_MP2_GeneSpecific_GenomeWide'] = pp_ex_OP['Points_MP2_GeneSpecific'].fillna(pp_ex_OP['Points_MP2_GenomeWide']).fillna(0)

In [ ]:
#calculate total points for Genome wide calibrations

#REVEL
pp_ex_OP['Total_Points_GenomeWide_REVEL'] = (
    pp_ex_OP['Fxn_points'].fillna(0) +
    pp_ex_OP['Points_REVEL_GenomeWide'].fillna(0)
)

#AM
pp_ex_OP['Total_Points_GenomeWide_AM'] = (
    pp_ex_OP['Fxn_points'].fillna(0) +
    pp_ex_OP['Points_AM_GenomeWide'].fillna(0)
)

#MP2
pp_ex_OP['Total_Points_GenomeWide_MP2'] = (
    pp_ex_OP['Fxn_points'].fillna(0) +
    pp_ex_OP['Points_MP2_GenomeWide'].fillna(0)
)

In [ ]:
#calculate total points for Gene Sepcific calibrations and default to GenomeWide calibrations when unavailable

#REVEL
pp_ex_OP['Total_Points_GeneSpecific_REVEL'] = (
    pp_ex_OP['Fxn_points'].fillna(0) +
    pp_ex_OP['Points_REVEL_GeneSpecific_GenomeWide'].fillna(0)
)

#AM
pp_ex_OP['Total_Points_GeneSpecific_AM'] = (
    pp_ex_OP['Fxn_points'].fillna(0) +
    pp_ex_OP['Points_AM_GeneSpecific_GenomeWide'].fillna(0)
)

#MP2
pp_ex_OP['Total_Points_GeneSpecific_MP2'] = (
    pp_ex_OP['Fxn_points'].fillna(0) +
    pp_ex_OP['Points_MP2_GeneSpecific_GenomeWide'].fillna(0)
)



In [ ]:
#calculate total points for OddsPath calibrations with GenomeWide calibrations

#REVEL
pp_ex_OP['Total_Points_OP_GenomeWide_REVEL'] = (
    pp_ex_OP['OP_points'].fillna(0) +
    pp_ex_OP['Points_REVEL_GenomeWide'].fillna(0)
)

#AM
pp_ex_OP['Total_Points_OP_GenomeWide_AM'] = (
    pp_ex_OP['OP_points'].fillna(0) +
    pp_ex_OP['Points_AM_GenomeWide'].fillna(0)
)

#MP2
pp_ex_OP['Total_Points_OP_GenomeWide_MP2'] = (
    pp_ex_OP['OP_points'].fillna(0) +
    pp_ex_OP['Points_MP2_GenomeWide'].fillna(0)
)


In [ ]:
def points_class(df, x, suffix):
    pts = df[x]
    valid = pts.notna()
    # placeholder 0 for invalid rows -- they're masked back to NaN below regardless
    pts_int = pts.where(valid, 0).astype(float).astype(int)

    conditions = [
        pts_int >= 10,
        (pts_int >= 6) & (pts_int <= 9),
        (pts_int >= 0) & (pts_int <= 5),
        (pts_int >= -6) & (pts_int <= -1),
        pts_int <= -7,
    ]
    choices = ['Pathogenic', 'Likely Pathogenic', 'Uncertain', 'Likely Benign', 'Benign']

    classifications = np.select(conditions, choices, default="__NA__")
    df[f"Class{suffix}"] = pd.Series(classifications, index=df.index).mask(
        ~valid | (classifications == "__NA__"), np.nan
    )

    return df

In [ ]:
points_class(pp_ex_OP, "Total_Points_GenomeWide_REVEL", "_GenomeWide_REVEL")
points_class(pp_ex_OP, "Total_Points_GenomeWide_AM", "_GenomeWide_AM")
points_class(pp_ex_OP, "Total_Points_GenomeWide_MP2", "_GenomeWide_MP2")

points_class(pp_ex_OP, "Total_Points_GeneSpecific_REVEL", "_GeneSpecific_REVEL")
points_class(pp_ex_OP, "Total_Points_GeneSpecific_AM", "_GeneSpecific_AM")
points_class(pp_ex_OP, "Total_Points_GeneSpecific_MP2", "_GeneSpecific_MP2")

points_class(pp_ex_OP, "Total_Points_OP_GenomeWide_REVEL", "OP_GenomeWide_REVEL")
points_class(pp_ex_OP, "Total_Points_OP_GenomeWide_AM", "OP_GenomeWide_AM")
points_class(pp_ex_OP, "Total_Points_OP_GenomeWide_MP2", "OP_GenomeWide_MP2")


In [ ]:
def split_zero(df, func, pred, total, suffix):
    f = df[func]
    p = df[pred]

    f_valid = f.notna()
    p_valid = p.notna()
    # placeholder 0 for invalid rows -- excluded from the conflict check via the *_valid masks
    f_int = f.where(f_valid, 0).astype(float).astype(int)
    p_int = p.where(p_valid, 0).astype(float).astype(int)

    both_na = ~f_valid & ~p_valid
    conflicting = f_valid & p_valid & (
        ((f_int > 0) & (p_int < 0)) | ((f_int < 0) & (p_int > 0))
    )

    # Default to the total column's own value (matches the "not conflicting" /
    # "exactly one of f, p missing" branches, which both just keep `total`).
    split = df[total].astype(object).copy()
    split[conflicting.values] = "Conflicting evidence"
    split[both_na.values] = "No evidence"

    df[f"Conflicting{suffix}"] = split

    return df

In [ ]:
#GenomeWide 

split_zero(pp_ex_OP, "Fxn_points", "Points_REVEL_GenomeWide","Total_Points_GenomeWide_REVEL", 
           "_REVEL_GenomeWide")
split_zero(pp_ex_OP, "Fxn_points", "Points_AM_GenomeWide","Total_Points_GenomeWide_AM", 
           "_AM_GenomeWide")
split_zero(pp_ex_OP, "Fxn_points", "Points_MP2_GenomeWide","Total_Points_GenomeWide_MP2", 
           "_MP2_GenomeWide")

#GeneSpecific

split_zero(pp_ex_OP, "Fxn_points", "Points_REVEL_GeneSpecific_GenomeWide","Total_Points_GeneSpecific_REVEL", 
           "_REVEL_GeneSpecific")
split_zero(pp_ex_OP, "Fxn_points", "Points_AM_GeneSpecific_GenomeWide","Total_Points_GeneSpecific_AM", 
           "_AM_GeneSpecific")
split_zero(pp_ex_OP, "Fxn_points", "Points_MP2_GeneSpecific_GenomeWide","Total_Points_GeneSpecific_MP2", 
           "_MP2_GeneSpecific")

#OddsPath

split_zero(pp_ex_OP, "OP_points", "Points_REVEL_GenomeWide","Total_Points_OP_GenomeWide_REVEL", 
           "_OP_REVEL_GenomeWide")
split_zero(pp_ex_OP, "OP_points", "Points_AM_GenomeWide","Total_Points_OP_GenomeWide_AM", 
           "_OP_AM_GenomeWide")
split_zero(pp_ex_OP, "OP_points", "Points_MP2_GenomeWide","Total_Points_OP_GenomeWide_MP2", 
           "_OP_MP2_GenomeWide")


In [ ]:
#remove F9 and TP53 datasets that are not the meta analysis

pp_ex_OP = pp_ex_OP[~pp_ex_OP['Dataset'].isin(['TP53_Boettcher_2019',
       'TP53_Fortuno_2021',
       'TP53_Giacomelli_2018_combined_score',
       'TP53_Giacomelli_2018_p53WT_Nutlin3',
       'TP53_Giacomelli_2018_p53null_Nutlin3',
       'TP53_Giacomelli_2018_p53null_etoposide', 'TP53_Kato_2003_AIP1nWT',
       'TP53_Kato_2003_BAXnWT', 'TP53_Kato_2003_GADD45nWT',
       'TP53_Kato_2003_MDM2nWT', 'TP53_Kato_2003_NOXAnWT',
       'TP53_Kato_2003_P53R2nWT', 'TP53_Kato_2003_WAF1nWT',
       'TP53_Kato_2003_h1433snWT','F9_Popp_2025_carboxy_F9_specific',
       'F9_Popp_2025_carboxy_gla_motif', 'F9_Popp_2025_heavy_chain',
       'F9_Popp_2025_light_chain',
       'F9_Popp_2025_strep_2'])]

In [ ]:
sankey = pp_ex_OP

In [ ]:
import numpy as np

cond2 = sankey[
    ['spliceAI_DS_AG', 'spliceAI_DS_AL', 'spliceAI_DS_DG', 'spliceAI_DS_DL']
].ge(0.2).any(axis=1)

cond3 = sankey['simplified_consequence'] == 'splice_site_variant'

sankey['splice_variant'] = np.where(cond2|cond3, 'Yes', 'No')

In [ ]:
#mark splice variants and ensure we don't use them in assays that can't detect splicing effects 

sankey['VariantNotes'] = np.where(
    (sankey['splice_variant'] == 'Yes') & (sankey['splice_measure'] != 'Yes'),
    'splice_variant_not_measured',
    ''
)

In [ ]:
#mark start lost variants and ensure we don't use them in assays that only measure at the amino acid level 

start_lost = (
    (sankey['nucleotide_or_aa'] == 'aa') &
    (sankey['simplified_consequence'] == 'start_lost')
)

tag = 'start_lost_variant_not_measured'

existing = sankey.loc[start_lost, 'VariantNotes'].fillna('').astype(str)

sankey.loc[start_lost, 'VariantNotes'] = np.where(
    existing != "",
    existing + ';' + tag,
    tag
)


In [ ]:
#split dataframe into nucleotide and protein level assays, different things need to happen for analysis
sankey_nuc = sankey[sankey['nucleotide_or_aa'] == 'nt']

sankey_aa = sankey[sankey['nucleotide_or_aa'] == 'aa']

In [ ]:
#mark any variants that get the opposite evidence between two assays

import numpy as np

group_cols = ['Gene', 'Chrom', 'hg38_start', 'ref_allele', 'alt_allele']

sankey_nuc['Chrom'] = sankey_nuc['Chrom'].astype(str)
sankey_nuc['hg38_start'] = sankey_nuc['hg38_start'].astype(str)
sankey_nuc['ref_allele'] = sankey_nuc['ref_allele'].astype(str)
sankey_nuc['alt_allele'] = sankey_nuc['alt_allele'].astype(str)
sankey_nuc['Gene'] = sankey_nuc['Gene'].astype(str)


def has_opposite_signs(x):
    x = x.dropna()
    non_zero = x[x != 0]
    return (non_zero > 0).any() and (non_zero < 0).any()

#2018 clinvar conflicting

conflict_mask_18 = sankey_nuc.groupby(group_cols)['Fxn_points'] \
    .transform(lambda x: has_opposite_signs(x))

conflict_mask_18 = conflict_mask_18.fillna(False)

mask_2_18 = conflict_mask_18 & sankey_nuc['VariantNotes'].notna() & (sankey_nuc['VariantNotes'] != "")

sankey_nuc.loc[mask_2_18, 'VariantNotes'] = sankey_nuc.loc[mask_2_18, 'VariantNotes'] + ';conflicting_fxn_data'

sankey_nuc.loc[conflict_mask_18 & ~mask_2_18, 'VariantNotes'] = 'conflicting_fxn_data'

In [ ]:
#get the first variant with max functional points and mark it 
def get_first_abs_max_idx(x):
    # Treat NaN as 0
    x_filled = x.fillna(0)

    # Compute the max absolute value
    abs_max = x_filled.abs().max()

    # Find the FIRST index where abs value equals abs_max
    return x_filled[x_filled.abs() == abs_max].index[0]

idx_max_18 = sankey_nuc.groupby(group_cols)['Fxn_points'].apply(
    lambda x: get_first_abs_max_idx(x)
)

#restrict to rows where Fxn_use_variant is NA/empty
mask_na_18 = sankey_nuc['VariantNotes'].isna() | (sankey_nuc['VariantNotes'] == "")

#update only those rows
sankey_nuc.loc[idx_max_18[idx_max_18.isin(sankey_nuc[mask_na_18].index)], 'VariantNotes'] = 'First_max_fxn_pts'

In [ ]:
# find maximum predictor points and assign only to those rows
idx_max_nuc_p_revel = sankey_nuc.groupby(group_cols)['Points_REVEL_GenomeWide'].apply(
    lambda x: x.fillna(0)[x.fillna(0).abs() == x.fillna(0).abs().max()].index
).explode()

idx_max_nuc_p_YP = sankey_nuc.groupby(group_cols)['Points_REVEL_GeneSpecific_GenomeWide'].apply(
    lambda x: x.fillna(0)[x.fillna(0).abs() == x.fillna(0).abs().max()].index
).explode()

#index

idx_max_nuc_p_revel = pd.Index(idx_max_nuc_p_revel)

idx_max_nuc_p_YP = pd.Index(idx_max_nuc_p_YP)

# assign only to those rows
sankey_nuc.loc[idx_max_nuc_p_revel, 'GenomeWide_REVEL_max'] = 'max_pred_pts'

sankey_nuc.loc[idx_max_nuc_p_YP, 'GeneSpecific_REVEL_max'] = 'max_pred_pts'

In [ ]:
#mark max points for AM Berquist et al. and AM cluster/gene specific calibrations 

idx_max_nuc_p_AM = sankey_nuc.groupby(group_cols)['Points_AM_GenomeWide'].apply(
    lambda x: x.fillna(0)[x.fillna(0).abs() == x.fillna(0).abs().max()].index
).explode()

idx_max_nuc_p_AM_YP = sankey_nuc.groupby(group_cols)['Points_AM_GeneSpecific_GenomeWide'].apply(
    lambda x: x.fillna(0)[x.fillna(0).abs() == x.fillna(0).abs().max()].index
).explode()

# flatten to Index
idx_max_nuc_p_AM = pd.Index(idx_max_nuc_p_AM)
idx_max_nuc_p_AM_YP = pd.Index(idx_max_nuc_p_AM_YP)

# assign only to those rows
sankey_nuc.loc[idx_max_nuc_p_AM, 'GenomeWide_AM_max'] = 'max_pred_pts'
sankey_nuc.loc[idx_max_nuc_p_AM_YP, 'GeneSpecific_AM_max'] = 'max_pred_pts'

In [ ]:
#mark max points for MP2 Pejaver et al. 
idx_max_nuc_p_MP2 = sankey_nuc.groupby(group_cols)['Points_MP2_GenomeWide'].apply(
    lambda x: x.fillna(0)[x.fillna(0).abs() == x.fillna(0).abs().max()].index
).explode()
idx_max_nuc_p_MP2_YP = sankey_nuc.groupby(group_cols)['Points_MP2_GeneSpecific_GenomeWide'].apply(
    lambda x: x.fillna(0)[x.fillna(0).abs() == x.fillna(0).abs().max()].index
).explode()

# flatten to Index
idx_max_nuc_p_MP2 = pd.Index(idx_max_nuc_p_MP2)
idx_max_nuc_p_MP2_YP = pd.Index(idx_max_nuc_p_MP2_YP)

# assign only to those rows
sankey_nuc.loc[idx_max_nuc_p_MP2, 'GenomeWide_MP2_max'] = 'max_pred_pts'
sankey_nuc.loc[idx_max_nuc_p_MP2_YP, 'GeneSpecific_MP2_max'] = 'max_pred_pts'

In [ ]:
#moving on to amino acid assays now 

import numpy as np

sankey_aa['Ref_seq_transcript_ID_stripped'] = sankey_aa['RefSeq Transcript ID'].str.replace(r'\.\d+$', '', regex=True)

sankey_aa['aa_pos'] = pd.to_numeric(sankey_aa['aa_pos'], errors='coerce')
sankey_aa['aa_ref'] = sankey_aa['aa_ref'].astype(str) 
sankey_aa['aa_alt'] = sankey_aa['aa_alt'].astype(str)
sankey_aa['Gene'] = sankey_aa['Gene'].astype(str)
sankey_aa['Ref_seq_transcript_ID_stripped'] = sankey_aa['Ref_seq_transcript_ID_stripped'].astype(str)

group_cols_aa = ['Gene', 'aa_ref', 'aa_pos', 'aa_alt','Ref_seq_transcript_ID_stripped']

def has_opposite_signs(x):
    x = x.dropna()
    non_zero = x[x != 0]
    return (non_zero > 0).any() and (non_zero < 0).any()


#2018 clinvar conflicting 

conflict_mask_aa_18 = sankey_aa.groupby(group_cols_aa)['Fxn_points'] \
    .transform(lambda x: has_opposite_signs(x))

conflict_mask_aa_18 = conflict_mask_aa_18.fillna(False)

mask_aa_18 = conflict_mask_aa_18 & sankey_aa['VariantNotes'].notna() & (sankey_aa['VariantNotes'] != "")

sankey_aa.loc[mask_aa_18, 'VariantNotes'] = sankey_aa.loc[mask_aa_18, 'VariantNotes'] + ';conflicting_fxn_data'

sankey_aa.loc[conflict_mask_aa_18 & ~mask_aa_18, 'VariantNotes'] = 'conflicting_fxn_data'

In [ ]:
idx_max_aa_18 = sankey_aa.groupby(group_cols_aa)['Fxn_points'].apply(
    lambda x: x.fillna(0)[x.fillna(0).abs() == x.fillna(0).abs().max()].index
).explode()

# convert to Index
idx_max_aa_18 = pd.Index(idx_max_aa_18)

mask_na_aa_18 = sankey_aa['VariantNotes'].isna() | (sankey_aa['VariantNotes'] == "")

sankey_aa.loc[idx_max_aa_18.intersection(sankey_aa[mask_na_aa_18].index), 'VariantNotes'] = 'max_fxn_pts'

In [ ]:
idx_max_aa_p_revel = sankey_aa.groupby(group_cols_aa)['Points_REVEL_GenomeWide'].apply(
    lambda x: x.fillna(0)[x.fillna(0).abs() == x.fillna(0).abs().max()].index
).explode()

idx_max_aa_p_revel_YP = sankey_aa.groupby(group_cols_aa)['Points_REVEL_GeneSpecific_GenomeWide'].apply(
    lambda x: x.fillna(0)[x.fillna(0).abs() == x.fillna(0).abs().max()].index
).explode()

# flatten to Index
idx_max_aa_p_revel = pd.Index(idx_max_aa_p_revel)
idx_max_aa_p_revel_YP = pd.Index(idx_max_aa_p_revel_YP)

# assign only to those rows
sankey_aa.loc[idx_max_aa_p_revel, 'GenomeWide_REVEL_max'] = 'max_pred_pts'
sankey_aa.loc[idx_max_aa_p_revel_YP, 'GeneSpecific_REVEL_max'] = 'max_pred_pts'

In [ ]:
#AM

idx_max_aa_p_AM = sankey_aa.groupby(group_cols_aa)['Points_AM_GenomeWide'].apply(
    lambda x: x.fillna(0)[x.fillna(0).abs() == x.fillna(0).abs().max()].index
).explode()

idx_max_aa_p_AM_YP = sankey_aa.groupby(group_cols_aa)['Points_AM_GeneSpecific_GenomeWide'].apply(
    lambda x: x.fillna(0)[x.fillna(0).abs() == x.fillna(0).abs().max()].index
).explode()

# flatten to Index
idx_max_aa_p_AM = pd.Index(idx_max_aa_p_AM)
idx_max_aa_p_AM_YP = pd.Index(idx_max_aa_p_AM_YP)

# assign only to those rows
sankey_aa.loc[idx_max_aa_p_AM, 'GenomeWide_AM_max'] = 'max_pred_pts'
sankey_aa.loc[idx_max_aa_p_AM_YP, 'GeneSpecific_AM_max'] = 'max_pred_pts'

In [ ]:
#MutPred2

idx_max_aa_p_mut = sankey_aa.groupby(group_cols_aa)['Points_MP2_GenomeWide'].apply(
    lambda x: x.fillna(0)[x.fillna(0).abs() == x.fillna(0).abs().max()].index
).explode()
idx_max_aa_p_mut_YP = sankey_aa.groupby(group_cols_aa)['Points_MP2_GeneSpecific_GenomeWide'].apply(
    lambda x: x.fillna(0)[x.fillna(0).abs() == x.fillna(0).abs().max()].index
).explode()

# flatten to Index
idx_max_aa_p_mut = pd.Index(idx_max_aa_p_mut)
idx_max_aa_p_mut_YP = pd.Index(idx_max_aa_p_mut_YP)

# assign only to those rows
sankey_aa.loc[idx_max_aa_p_mut, 'GenomeWide_MP2_max'] = 'max_pred_pts'
sankey_aa.loc[idx_max_aa_p_mut_YP, 'GeneSpecific_MP2_max'] = 'max_pred_pts'

In [ ]:
priority_genes = ['BRCA1', 'PTEN', 'MSH2', 'TP53']

#create new column with clinvar significance for genes where 2018 calibrations are needed, and if not then 2025 
sankey_aa['clinvar_18_25'] = np.where(
    sankey_aa['Gene'].isin(priority_genes),  
    sankey_aa['clinvar_sig_2018'], 
    sankey_aa['clinvar_sig_2025']
)


In [ ]:
pathogenic_group = {
    "Pathogenic", "Likely pathogenic", "Pathogenic/Likely pathogenic"
}
benign_group = {
    "Benign", "Likely benign", "Benign/Likely benign"
}

conflicts_group = {"Conflicting classifications of pathogenicity"}

def has_conflict(series):
    has_path = any(val in pathogenic_group for val in series)
    has_benign = any(val in benign_group for val in series)
    has_conflict_val = any(val in conflicts_group for val in series)
    return (has_path and has_benign) or (has_path and has_conflict_val) or (has_benign and has_conflict_val)


sankey_aa['clinvar_conflict_flag_18_25'] = (
    sankey_aa.groupby(group_cols_aa)['clinvar_18_25']
    .transform(lambda x: 'has clinvar conflict' if has_conflict(x) else np.nan)
)

In [ ]:
# Define your groups
pathogenic_group = {
    "Pathogenic", "Likely pathogenic", "Pathogenic/Likely pathogenic"
}
benign_group = {
    "Benign", "Likely benign", "Benign/Likely benign"
}
conflicts_group = {"Conflicting classifications of pathogenicity"}

def summarize_clnsig(series):
    sigs = set(series.dropna())  # unique values, ignore NaN/None

    # If everything is NA → return blank
    if len(sigs) == 0:
        return "Unseen"

    # Explicit flags
    has_path = any(val in pathogenic_group for val in sigs)
    has_benign = any(val in benign_group for val in sigs)
    has_conflict = any(val in conflicts_group for val in sigs)
    has_uncertain = "Uncertain significance" in sigs

    # Conflict rules
    if has_conflict or (has_path and has_benign):
        return "has clinvar conflict"

    # Pathogenic-only group
    if has_path:
        if "Likely pathogenic" in sigs or "Pathogenic/Likely pathogenic" in sigs:
            return "Likely pathogenic"
        return "Pathogenic"

    # Benign-only group
    if has_benign:
        if "Likely benign" in sigs or "Benign/Likely benign" in sigs:
            return "Likely benign"
        return "Benign"

    # Uncertain-only group
    if has_uncertain and len(sigs) == 1:
        return "Uncertain significance"

    # If it's a mix of uncertain with something else → conflict
    if has_uncertain:
        return "VUS/conflict"

    if len(sigs) == 1:
        return next(iter(sigs))   # grab the single element from the set
    else:
        return "multiple other classifications"


# Apply per group and broadcast back

sankey_aa["clnsig_group_18_25"] = (
    sankey_aa.groupby(group_cols_aa)["clinvar_18_25"]
    .transform(summarize_clnsig)
)

In [ ]:
def splice_var(series):
    sigs = set(series.dropna())

    allowed = {"Yes", "No"}
    invalid = sigs - allowed
    if invalid:
        raise ValueError(f"Invalid splice_variant values found: {invalid}")

    return "Yes" if "Yes" in sigs else "No"

        
sankey_aa["splice_var_amino"] = (
    sankey_aa.groupby(group_cols_aa)["splice_variant"]
    .transform(splice_var)
)  

In [ ]:
def revel_train_var(series):
    sigs = set(series.dropna())

    if len(sigs) == 0:
        return ""          
    elif True in sigs: 
        return "Yes"
    else:
        return "No"

sankey_aa["revel_train_amino"] = (
    sankey_aa.groupby(group_cols_aa)["REVEL_train"]
    .transform(revel_train_var)
)  

In [ ]:
def mutpred2_train_var(series):
    sigs = set(series.dropna())

    if len(sigs) == 0:
        return ""          
    elif True in sigs:   
        return "Yes"
    else:                  
        return "No"

        
sankey_aa["mp2_train_amino"] = (
    sankey_aa.groupby(group_cols_aa)["MP2_train"]
    .transform(mutpred2_train_var)
)      

In [ ]:
import numpy as np

priority_genes = ['BRCA1', 'PTEN', 'MSH2', 'TP53']

#create new column with clinvar significance for genes where 2018 calibrations are needed, and if not then 2025 
sankey_nuc['clinvar_18_25'] = np.where(
    sankey_nuc['Gene'].isin(priority_genes),  
    sankey_nuc['clinvar_sig_2018'], 
    sankey_nuc['clinvar_sig_2025']
)

# clnsig_group just copies clinvar_sig
sankey_nuc['clnsig_group_25'] = sankey_nuc['clinvar_sig_2025']
sankey_nuc['clnsig_group_18_25'] = sankey_nuc['clinvar_18_25']

# revel_train_amino: Yes if True, else No
sankey_nuc['revel_train_amino'] = np.where(
    sankey_nuc['REVEL_train'] == True, "Yes", "No"
)

# mp2_train_amino: Yes if True, else No
sankey_nuc['mp2_train_amino'] = np.where(
    sankey_nuc['MP2_train'] == True, "Yes", "No"
)

sankey_nuc['splice_var_amino'] = sankey_nuc['splice_variant']

In [ ]:
import pandas as pd
sankey_f = pd.concat([sankey_nuc, sankey_aa])

In [ ]:
group_cols = ['Gene', 'Chrom', 'hg38_start', 'ref_allele', 'alt_allele']

sankey_f['Chrom'] = sankey_f['Chrom'].astype(str)
sankey_f['ref_allele'] = sankey_f['ref_allele'].astype(str)
sankey_f['alt_allele'] = sankey_f['alt_allele'].astype(str)
sankey_f['Gene'] = sankey_f['Gene'].astype(str)
sankey_f['Fxn_points'] = pd.to_numeric(sankey_f['Fxn_points'], errors='coerce')
sankey_f['hg38_start'] = pd.to_numeric(sankey_f['hg38_start'], errors = 'coerce')


def has_opposite_signs(x):
    x = x.dropna()
    non_zero = x[x != 0]
    return (non_zero > 0).any() and (non_zero < 0).any()

conflicting_rows_full_18 = sankey_f.groupby(group_cols).filter(lambda x: has_opposite_signs(x['Fxn_points']))

id_list_18 = list(conflicting_rows_full_18['mavedb_variant_urn'])

disallowed = [
    "splice_variant_not_measured",
    "splice_variant_not_measured;conflicting_fxn_data","conflicting_fxn_data"
]

mask_conf_18 = (
    sankey_f['mavedb_variant_urn'].isin(id_list_18) &
    ~sankey_f['VariantNotes'].isin(disallowed)
)

sankey_f.loc[mask_conf_18, 'VariantNotes'] = "conflicting_fxn_data"


In [ ]:
sankey_g = sankey_f

In [ ]:
sankey_g = sankey_g.drop(columns = ['dataset_2025', 'prior_2025', 'relax_2025', 'n_c_2025', 'benign_method_2025', 
                         'clinvar_2018_2025', 'scoreset_flipped_2025', 'range_-8_2025', 'range_-7_2025', 
                         'range_-6_2025', 'range_-5_2025', 'range_-4_2025', 'range_-3_2025', 'range_-2_2025', 
                         'range_-1_2025', 'range_1_2025', 'range_2_2025', 'range_3_2025', 'range_4_2025', 'range_5_2025', 
                         'range_6_2025', 'range_7_2025', 'range_8_2025', 'base', 'dataset_2018', 'prior_2018', 
                         'relax_2018', 'n_c_2018', 'benign_method_2018', 'clinvar_2018_2018', 'scoreset_flipped_2018', 
                         'range_-8_2018', 'range_-7_2018', 'range_-6_2018', 'range_-5_2018', 'range_-4_2018', 'range_-3_2018', 
                         'range_-2_2018', 'range_-1_2018', 'range_1_2018', 'range_2_2018', 'range_3_2018', 'range_4_2018', 
                         'range_5_2018', 'range_6_2018', 'range_7_2018', 'range_8_2018','Total Controls','Pathogenic Controls', 
                         'Benign Controls', 'Prior Probability Pathogenic', 'Total Assay Abnormal', 'True Path in Abnormal', 
                         'Total Assay Normal', 'True Path in Normal', 'Pseudocount Details', 'Evidence Code Normal', 
                         'Evidence Code Abnormal','Dataset Name', 'Score Intervals Reported?', 'Functional Classification Provided?',
                         'Ref_seq_transcript_ID_stripped'])
                         

In [ ]:
sankey_g.to_csv(RECLASSIFICATION_DIR/"integrated_variant_effect_dataset_analysis.csv.gz", compression = 'gzip', index = None)

In [ ]:
import pandas as pd

sankey_f = pd.read_csv(RECLASSIFICATION_DIR/"integrated_variant_effect_dataset_analysis.csv.gz")

In [ ]:
sankey_f = sankey_f[sankey_f['Gene'] != 'SFPQ']

In [ ]:
#flag CHEK2 variants that need to be filtered for any clinical analysis

chek2 = pd.read_excel(INPUT_MAVES_DIR/"CHEK2_Gebbia_2024.xlsx", header = 0)

In [ ]:
# hgvs_p carries a RefSeq protein accession prefix (e.g. 'NP_009125.1:p.Ser2Ala')
# that chek2['hgvs_pro'] doesn't have (just 'p.Ser2Ala') -- matching on hgvs_p
# directly against hgvs_pro therefore never matches anything, silently making this
# QC exclusion a no-op. Strip the prefix before matching.
sankey_f['_hgvs_p_no_transcript'] = sankey_f['hgvs_p'].str.replace(r'^[^:]+:', '', regex=True)

tmp = pd.merge(sankey_f, chek2, left_on = ['_hgvs_p_no_transcript', 'auth_reported_score'],
               right_on = ['hgvs_pro','score'], how = 'left')

In [ ]:
import numpy as np
tmp['Flag'] = np.where(tmp['Filter_CI'] == 1, '*', tmp['Flag'])

In [ ]:
sankey_f = tmp.drop(columns = ['aaChange', 'hgvs_pro', '_hgvs_p_no_transcript',
       'type', 'score', 'error', 'LLR', 'LLR_strength', 'Filter_CI',
       'Filter_Hypercomplement'])

In [ ]:
#take out conflicting functional data and splice variants that are not measured 

dis = ['conflicting_fxn_data',
       'splice_variant_not_measured',
       'splice_variant_not_measured;conflicting_fxn_data','start_lost_variant_not_measured']

sankey_f = sankey_f[
    ~sankey_f['VariantNotes'].isin(dis) &
    ((sankey_f['splice_var_amino'] != 'Yes') | (sankey_f['splice_measure'] == 'Yes'))
]

In [ ]:
#remove flagged variants
sankey_f = sankey_f[sankey_f['Flag'] != '*']

In [ ]:
#controls 

controls = sankey_f[sankey_f['clnsig_group_18_25'].isin(['Benign','Benign/Likely benign','Likely benign','Pathogenic',
                                       'Pathogenic/Likely pathogenic','Likely pathogenic'])]

In [ ]:
import numpy as np

one_plus_stars = [
    'criteria provided, single submitter',
    'criteria provided, multiple submitters, no conflicts',
    'reviewed by expert panel',
    'criteria provided, conflicting classifications'
]

priority_genes = ['BRCA1', 'PTEN', 'MSH2', 'TP53']

#create new column with clinvar stars for genes where 2018 calibrations are needed, and if not then 2025 
controls['clinvar_star_18_25'] = np.where(
    controls['Gene'].isin(priority_genes),  
    controls['clinvar_star_2018'], 
    controls['clinvar_star_2025']
)

# Mixed-year ClinVar review date, mirroring clinvar_star_18_25's own
# per-gene 2018-vs-2025 vintage switch -- used as the aa-stage recency
# tie-break below.
controls['clinvar_date_last_reviewed_18_25'] = np.where(
    controls['Gene'].isin(priority_genes),
    controls['clinvar_date_last_reviewed_2018'],
    controls['clinvar_date_last_reviewed_2025'],
)

In [ ]:
#remove clinvar conflicts and splice variants 

controls = controls[(controls['clinvar_conflict_flag_18_25'] != 'has clinvar conflict') & 
((controls['splice_var_amino'] != 'Yes') | (controls['splice_measure'] == 'Yes'))]

In [ ]:
# --- Deduplication strategy for variant categories scored by more than one
# assay. Two independently-configurable strategies -- see
# docs/variant_classification.md for the full rationale:
#
# `CONTROLS_CLINGEN_DEDUP_STRATEGY` -- governs ClinVar controls and ClinGen
# Evidence Repo controls (the `controls_aa`/`clingen_aa` aa-level tie-break
# and the nt+aa merge below, `catch_mis_2`). These categories must avoid
# double-counting the same underlying variant's evidence at both the DNA and
# amino-acid level, so DNA-resolution (nt) evidence is preferred outright
# over amino-acid-resolution (aa) evidence.
#
# `VUS_GNOMAD_UNOBSERVED_DEDUP_STRATEGY` -- governs VUS, gnomAD, and
# Unobserved variants (`dedup_vus_gnomad_unobserved`, defined below). These
# categories are evaluated at DNA resolution only, so double-counting across
# resolutions isn't a concern -- the record with the greatest absolute
# points wins outright.
#
# Both parameters take the same three values (see each function's docstring
# for the exact per-category mechanics, which differ):
#   "v1"               - each category's original, pre-parameter behavior.
#                        For controls/ClinGen: aa-level ties broken by
#                        ASSAY_PRIORITY_LIST rank, nt+aa merge broken by
#                        signed Fxn_points. For VUS/gnomAD/Unobserved: sorted
#                        by VariantNotes tag, which has an accidental
#                        nt-over-aa bias (see
#                        docs/assay_priority_questions.md)
#   "abs_max"          - greatest absolute Fxn_points wins, nt and aa
#                        candidates treated identically
#   "nt_then_abs_max"  - an nt-assay record always beats an aa-assay record
#                        for the same variant; ties within a type broken by
#                        greatest absolute Fxn_points
CONTROLS_CLINGEN_DEDUP_STRATEGY = "nt_then_abs_max"  # "v1" | "abs_max" | "nt_then_abs_max"
VUS_GNOMAD_UNOBSERVED_DEDUP_STRATEGY = "abs_max"  # "v1" | "abs_max" | "nt_then_abs_max"

assert CONTROLS_CLINGEN_DEDUP_STRATEGY in ("v1", "abs_max", "nt_then_abs_max")
assert VUS_GNOMAD_UNOBSERVED_DEDUP_STRATEGY in ("v1", "abs_max", "nt_then_abs_max")

In [ ]:
# aa-stage SNV-representative tie-break for controls/ClinGen_Repo, the nt+aa
# merge (catch_mis_2), and the VUS/gnomAD/Unobserved dedup below all live in
# src/lib/dedup.py -- shared with OddsPath_classifications.ipynb so a fix to
# one pipeline's dedup mechanics can't silently drift out of sync with the
# other's, the way OddsPath_classifications.ipynb's own independent copies
# previously did (still-signed-value catch_mis_2, pure-assay_priority
# aa-stage tie-break with no keep-all, no VUS/gnomAD/Unobserved Dataset-name
# fallback). See docs/variant_classification.md's "Decided approach" section
# for the full rationale.
from src.lib.dedup import (
    CLINVAR_REVIEW_STATUS_RANK,
    GENOMIC_KEY_COLS,
    aa_dedup_or_mark,
    catch_mis_2,
    clingen_aa_sort_key,
    controls_aa_sort_key,
    dedup_vus_gnomad_unobserved,
)


In [ ]:
controls_nuc = controls[controls['nucleotide_or_aa'] == 'nt']

controls_aa = controls[controls['nucleotide_or_aa'] == 'aa']

In [ ]:
controls_nuc_drop = (controls_nuc
    .sort_values(by="VariantNotes", na_position="last") 
    .drop_duplicates(subset=['Gene', 'hg38_start', 'ref_allele', 'alt_allele'], keep="first")
)

In [ ]:
controls_nuc_drop = controls_nuc_drop[controls_nuc_drop['clinvar_star_18_25'].isin(one_plus_stars)]

In [ ]:
from src.lib.assay_priority_v1 import ASSAY_PRIORITY_LIST

assay_priority_list = ASSAY_PRIORITY_LIST
assay_priority_map = {name: i for i, name in enumerate(assay_priority_list)}

controls_aa["assay_priority"] = controls_aa["Dataset"].map(assay_priority_map)

controls_aa["assay_priority"] = controls_aa["assay_priority"].fillna(9999)

In [ ]:
controls_aa['Ref_seq_transcript_ID_stripped'] = controls_aa['RefSeq Transcript ID'].str.replace(r'\.\d+$', '', regex=True)

group_cols_aa_cln = ["Gene", "aa_pos", "aa_ref", "aa_alt","Ref_seq_transcript_ID_stripped"]

one_plus_stars = {
    'criteria provided, single submitter',
    'criteria provided, multiple submitters, no conflicts',
    'reviewed by expert panel',
    'criteria provided, conflicting classifications'
}

zero_stars = {
    'no classification for the single variant',
    'no classification provided','no assertion criteria provided'
}


def summarize_clnstar(series):
    sigs = set(series.dropna())

    # All missing
    if len(sigs) == 0:
        return "Unseen"

    one_star = any(val in one_plus_stars for val in sigs)
    zero_star = any(val in zero_stars for val in sigs)

    if one_star and zero_star:
        return "has_clinvar_star_conflict"

    if one_star:
        return "one_plus_star"

    if zero_star:
        return "zero_star"

    raise ValueError(
        f"Unexpected ClinVar review_status values encountered: {sigs}"
    )

controls_aa["clinvar_star_18_25_group"] = (
    controls_aa
    .groupby(group_cols_aa_cln)["clinvar_star_18_25"]
    .transform(summarize_clnstar)
)

In [ ]:
controls_aa = controls_aa[controls_aa['clinvar_star_18_25_group'].isin(['one_plus_star'])]

In [ ]:
group_cols_aa_dedup = ["Gene", "aa_pos", "aa_ref", "aa_alt", "Ref_seq_transcript_ID_stripped"]

aa_sort_by, aa_ascending = controls_aa_sort_key(controls_aa, CONTROLS_CLINGEN_DEDUP_STRATEGY)

#REVEL
controls_aa_drop_REVEL_YP = aa_dedup_or_mark(
    controls_aa[
        (controls_aa['VariantNotes'] == 'max_fxn_pts')
        & (controls_aa['GeneSpecific_REVEL_max'] == 'max_pred_pts')
    ],
    group_cols_aa_dedup, GENOMIC_KEY_COLS, aa_sort_by, aa_ascending, CONTROLS_CLINGEN_DEDUP_STRATEGY,
)

#MP2

controls_aa_drop_mut_YP = aa_dedup_or_mark(
    controls_aa[
        (controls_aa['VariantNotes'] == 'max_fxn_pts')
        & (controls_aa['GeneSpecific_MP2_max'] == 'max_pred_pts')
    ],
    group_cols_aa_dedup, GENOMIC_KEY_COLS, aa_sort_by, aa_ascending, CONTROLS_CLINGEN_DEDUP_STRATEGY,
)


#AM
controls_aa_drop_AM_YP = aa_dedup_or_mark(
    controls_aa[
        (controls_aa['VariantNotes'] == 'max_fxn_pts')
        & (controls_aa['GeneSpecific_AM_max'] == 'max_pred_pts')
    ],
    group_cols_aa_dedup, GENOMIC_KEY_COLS, aa_sort_by, aa_ascending, CONTROLS_CLINGEN_DEDUP_STRATEGY,
)


In [ ]:
#REVEL

controls_no_dup_REVEL_YP = pd.concat([controls_nuc_drop,controls_aa_drop_REVEL_YP])

#MP2


controls_no_dup_mut_YP = pd.concat([controls_nuc_drop,controls_aa_drop_mut_YP])

#AM

controls_no_dup_AM_YP = pd.concat([controls_nuc_drop,controls_aa_drop_AM_YP])



In [ ]:
#remove training variants from controls and duplicates on the nucleotide level!

controls_no_dup_REVEL_YP = controls_no_dup_REVEL_YP[controls_no_dup_REVEL_YP['revel_train_amino'] != 'Yes']

controls_no_dup_mut_YP = controls_no_dup_mut_YP[controls_no_dup_mut_YP['mp2_train_amino'] != 'Yes']


In [ ]:
#REVEL

group_cols = ['Gene', 'Chrom', 'hg38_start', 'ref_allele', 'alt_allele']


controls_no_dup_REVEL_YP_cleaned  = catch_mis_2(
    controls_no_dup_REVEL_YP,
    group_cols, points_col='Fxn_points', strategy=CONTROLS_CLINGEN_DEDUP_STRATEGY
)

controls_no_dup_MP2_YP_cleaned  = catch_mis_2(
    controls_no_dup_mut_YP,
    group_cols, points_col='Fxn_points', strategy=CONTROLS_CLINGEN_DEDUP_STRATEGY
)

controls_no_dup_AM_YP_cleaned  = catch_mis_2(
    controls_no_dup_AM_YP,
    group_cols, points_col='Fxn_points', strategy=CONTROLS_CLINGEN_DEDUP_STRATEGY
)

In [ ]:
#VUS, check all on the nucleotide level 

#sankey_f already has conflicting functional data, splice variants and Flags removed, need to remove training variants where appropriate

VUS = sankey_f[sankey_f['clinvar_sig_2025'].isin(['Uncertain significance'])]

In [ ]:
VUS_no_dup = dedup_vus_gnomad_unobserved(
    VUS,
    ['Gene', 'hg38_start', 'ref_allele', 'alt_allele'],
    strategy=VUS_GNOMAD_UNOBSERVED_DEDUP_STRATEGY,
)

In [ ]:
VUS_no_dup_REVEL = VUS_no_dup[VUS_no_dup['revel_train_amino'] != "Yes"]

In [ ]:
VUS_no_dup_mut= VUS_no_dup[VUS_no_dup['mp2_train_amino'] != "Yes"]

In [ ]:
VUS_no_dup_AM = VUS_no_dup

In [ ]:
# Unseen nucleotide variants, sankey_f is already filtered for splice variants not measured, conflicting functional data, and Flags removed, need to remove training variants where appropriate

Unseen = sankey_f[(sankey_f['clinvar_sig_2025'].isna()) & (sankey_f['gnomad_MAF'].isna())]

In [ ]:
unseen_no_dup = dedup_vus_gnomad_unobserved(
    Unseen,
    ['Gene', 'hg38_start', 'ref_allele', 'alt_allele'],
    strategy=VUS_GNOMAD_UNOBSERVED_DEDUP_STRATEGY,
)

In [ ]:
#filter for SNVs
unseen_no_dup = unseen_no_dup[
    (unseen_no_dup['ref_allele'].str.len() == 1) &
    (unseen_no_dup['alt_allele'].str.len() == 1)
]

In [ ]:
unseen_no_dup_REVEL = unseen_no_dup[unseen_no_dup['revel_train_amino'] != "Yes"]

In [ ]:
unseen_no_dup_mut = unseen_no_dup[unseen_no_dup['mp2_train_amino'] != "Yes"]

In [ ]:
unseen_no_dup_AM = unseen_no_dup

In [ ]:
#gnomad nucleotide variants, sankey_f is already filtered for splice variants not measured, conflicting functional data, and Flags removed, need to remove training variants where appropriate

gnomad = sankey_f[sankey_f['gnomad_MAF'].notna()]

In [ ]:
gnomad_no_dup = dedup_vus_gnomad_unobserved(
    gnomad,
    ['Gene', 'hg38_start', 'ref_allele', 'alt_allele'],
    strategy=VUS_GNOMAD_UNOBSERVED_DEDUP_STRATEGY,
)

In [ ]:
gnomad_no_dup_REVEL = gnomad_no_dup[gnomad_no_dup['revel_train_amino'] != "Yes"]

gnomad_no_dup_mut = gnomad_no_dup[gnomad_no_dup['mp2_train_amino'] != "Yes"]

gnomad_no_dup_AM = gnomad_no_dup


In [ ]:
clingen = sankey_f[(sankey_f['Updated_Classification_ClinGen_repo'].notna()) & (sankey_f['Updated_Classification_ClinGen_repo'] != 'VUS')]

In [ ]:
clingen_nuc = clingen[clingen['nucleotide_or_aa'] == 'nt']

clingen_aa = clingen[clingen['nucleotide_or_aa'] == 'aa']

In [ ]:
clingen_nuc_drop = (clingen_nuc
    .sort_values(by="VariantNotes", na_position="last") 
    .drop_duplicates(subset=['Gene', 'hg38_start', 'ref_allele', 'alt_allele'], keep="first")
)

In [ ]:
clingen_aa["assay_priority"] = clingen_aa["Dataset"].map(assay_priority_map)

clingen_aa["assay_priority"] = clingen_aa["assay_priority"].fillna(9999)

In [ ]:
clingen_aa['Ref_seq_transcript_ID_stripped'] = clingen_aa['RefSeq Transcript ID'].str.replace(r'\.\d+$', '', regex=True)

group_cols_aa_dedup = ["Gene", "aa_pos", "aa_ref", "aa_alt", "Ref_seq_transcript_ID_stripped"]

aa_sort_by, aa_ascending = clingen_aa_sort_key(clingen_aa, CONTROLS_CLINGEN_DEDUP_STRATEGY)

#REVEL
clingen_aa_drop_REVEL_YP = aa_dedup_or_mark(
    clingen_aa[
        (clingen_aa['VariantNotes'] == 'max_fxn_pts')
        & (clingen_aa['GeneSpecific_REVEL_max'] == 'max_pred_pts')
    ],
    group_cols_aa_dedup, GENOMIC_KEY_COLS, aa_sort_by, aa_ascending, CONTROLS_CLINGEN_DEDUP_STRATEGY,
)

#MP2

clingen_aa_drop_mut_YP = aa_dedup_or_mark(
    clingen_aa[
        (clingen_aa['VariantNotes'] == 'max_fxn_pts')
        & (clingen_aa['GeneSpecific_MP2_max'] == 'max_pred_pts')
    ],
    group_cols_aa_dedup, GENOMIC_KEY_COLS, aa_sort_by, aa_ascending, CONTROLS_CLINGEN_DEDUP_STRATEGY,
)

#AM

clingen_aa_drop_AM_YP = aa_dedup_or_mark(
    clingen_aa[
        (clingen_aa['VariantNotes'] == 'max_fxn_pts')
        & (clingen_aa['GeneSpecific_AM_max'] == 'max_pred_pts')
    ],
    group_cols_aa_dedup, GENOMIC_KEY_COLS, aa_sort_by, aa_ascending, CONTROLS_CLINGEN_DEDUP_STRATEGY,
)

In [ ]:
clingen_no_dup_REVEL_YP = pd.concat([clingen_nuc_drop,clingen_aa_drop_REVEL_YP])

clingen_no_dup_mut_YP = pd.concat([clingen_nuc_drop,clingen_aa_drop_mut_YP])

clingen_no_dup_AM_YP = pd.concat([clingen_nuc_drop,clingen_aa_drop_AM_YP])


In [ ]:
ClinGen_repo_REVEL_YP = clingen_no_dup_REVEL_YP[clingen_no_dup_REVEL_YP['revel_train_amino'] != "Yes"]

ClinGen_repo_mut_YP = clingen_no_dup_mut_YP[clingen_no_dup_mut_YP['mp2_train_amino'] != "Yes"]

ClinGen_repo_AM_YP = clingen_no_dup_AM_YP

In [ ]:
group_cols = ['Gene', 'Chrom', 'hg38_start', 'ref_allele', 'alt_allele']


ClinGen_repo_REVEL_YP_cleaned  = catch_mis_2(
    ClinGen_repo_REVEL_YP,
    group_cols, points_col='Fxn_points', strategy=CONTROLS_CLINGEN_DEDUP_STRATEGY
)

ClinGen_repo_MP2_YP_cleaned  = catch_mis_2(
    ClinGen_repo_mut_YP,
    group_cols, points_col='Fxn_points', strategy=CONTROLS_CLINGEN_DEDUP_STRATEGY
)

ClinGen_repo_AM_YP_cleaned  = catch_mis_2(
    ClinGen_repo_AM_YP,
    group_cols, points_col='Fxn_points', strategy=CONTROLS_CLINGEN_DEDUP_STRATEGY
)

In [ ]:
from datetime import datetime

today = datetime.today().strftime("%Y_%m_%d")


dfs = {
    "ClinGen_Repo_REVEL_GeneSpecific": ClinGen_repo_REVEL_YP_cleaned,
    "ClinGen_Repo_MP2_GeneSpecific": ClinGen_repo_MP2_YP_cleaned,
    "ClinGen_Repo_AM_GeneSpecific": ClinGen_repo_AM_YP_cleaned,

    
    "gnomAD_REVEL": gnomad_no_dup_REVEL,
    "gnomAD_AM": gnomad_no_dup_AM,
    "gnomAD_MP2": gnomad_no_dup_mut,

    
    "Unobserved_REVEL" : unseen_no_dup_REVEL,
    "Unobserved_AM" : unseen_no_dup_AM,
    "Unobserved_MP2" : unseen_no_dup_mut,

    
    "VUS_REVEL" : VUS_no_dup_REVEL,
    "VUS_AM" : VUS_no_dup_AM,
    "VUS_MP2" : VUS_no_dup_mut,


    "controls_REVEL_GeneSpecific": controls_no_dup_REVEL_YP_cleaned,
    "controls_AM_GeneSpecific": controls_no_dup_AM_YP_cleaned,
    "controls_MP2_GeneSpecific": controls_no_dup_MP2_YP_cleaned,

    

}

In [ ]:
# controls_aa/clingen_aa (non-"v1" strategies) already tagged their rows
# Variant_Role="primary"/"secondary"; nt-origin rows within those same
# categories never went through that aa collision logic (an nt row is
# always the sole representative for its own coordinate) and so are NaN
# after the nt+aa concat -- fill them "primary". Every other category
# (VUS/gnomAD/Unobserved, untouched by this change; "v1"-strategy
# controls/ClinGen_Repo, which drops losers outright) never had the
# column at all -- create it fresh as "primary" for schema consistency
# across all 15 output sheets.
for _name, _df in dfs.items():
    if "Variant_Role" in _df.columns:
        dfs[_name] = _df.assign(Variant_Role=_df["Variant_Role"].fillna("primary"))
    else:
        dfs[_name] = _df.assign(Variant_Role="primary")


In [ ]:
COLUMNS_TO_DROP = [
    'REVEL_GenomeWide_Code', 'MP2_GenomeWide_Code', 'AM_GenomeWide_Code', 'REVEL_GeneSpecific_Code', 'AM_GeneSpecific_Code', 
    'MP2_GeneSpecific_Code', 'Points_REVEL_GenomeWide', 'Points_REVEL_GeneSpecific', 'Points_AM_GenomeWide', 'Points_AM_GeneSpecific', 
    'Points_MP2_GenomeWide', 'Points_MP2_GeneSpecific','Total_Points_GenomeWide_REVEL', 'Total_Points_GenomeWide_AM', 
    'Total_Points_GenomeWide_MP2','Total_Points_OP_GenomeWide_REVEL', 'Total_Points_OP_GenomeWide_AM', 'Total_Points_OP_GenomeWide_MP2', 
    'Class_GenomeWide_REVEL', 'Class_GenomeWide_AM', 'Class_GenomeWide_MP2','ClassOP_GenomeWide_REVEL', 'ClassOP_GenomeWide_AM', 
    'ClassOP_GenomeWide_MP2', 'Conflicting_REVEL_GenomeWide', 'Conflicting_AM_GenomeWide', 'Conflicting_MP2_GenomeWide',
    'Conflicting_OP_REVEL_GenomeWide', 'Conflicting_OP_AM_GenomeWide', 'Conflicting_OP_MP2_GenomeWide', 'splice_variant', 
    'VariantNotes', 'GenomeWide_REVEL_max', 'GeneSpecific_REVEL_max', 'GenomeWide_AM_max', 'GeneSpecific_AM_max', 'GenomeWide_MP2_max', 
    'GeneSpecific_MP2_max', 'clnsig_group_25','revel_train_amino', 'mp2_train_amino', 'splice_var_amino', 'clinvar_conflict_flag_18_25', 
    'clinvar_star_18_25', 'assay_priority', 'Ref_seq_transcript_ID_stripped', 'clinvar_star_18_25_group'   
]

In [ ]:
DROP_DFS = {
    "ClinGen_Repo_REVEL_GeneSpecific",
    "ClinGen_Repo_MP2_GeneSpecific",
    "ClinGen_Repo_AM_GeneSpecific",
    
    "gnomAD_REVEL",
    "gnomAD_AM",
    "gnomAD_MP2",
    
    "Unobserved_REVEL",
    "Unobserved_AM",
    "Unobserved_MP2",
    
    "VUS_REVEL",
    "VUS_AM",
    "VUS_MP2",
    
    "controls_REVEL_GeneSpecific",
    "controls_AM_GeneSpecific",  
    "controls_MP2_GeneSpecific",

}


In [ ]:
dfs_final = {}

for name, df in dfs.items():
    if name in DROP_DFS:
        dfs_final[name] = df.drop(columns=COLUMNS_TO_DROP, errors="ignore")
    else:
        dfs_final[name] = df


In [ ]:
# Define new column names
rename_dict = {
    'Total_Points_GeneSpecific_REVEL': 'Total_Points_REVEL',
    'Total_Points_GeneSpecific_AM': 'Total_Points_AM',
    'Total_Points_GeneSpecific_MP2': 'Total_Points_MP2',
    'Class_GeneSpecific_REVEL': 'Class_REVEL', 
    'Class_GeneSpecific_AM':'Class_AM',
    'Class_GeneSpecific_MP2':'Class_MP2',
    'Conflicting_REVEL_GeneSpecific': 'Conflicting_REVEL',
    'Conflicting_AM_GeneSpecific': 'Conflicting_AM',
    'Conflicting_MP2_GeneSpecific': 'Conflicting_MP2',
    'clinvar_18_25': 'clinvar_sig_18_25'

}

# Define column order
column_order = ['mavedb_variant_urn', 'Dataset', 'Gene', 'HGNC_id', 'Chrom', 'Strand', 'hg38_start', 'hg38_end', 'ref_allele', 'alt_allele', 
                'auth_transcript_id', 'transcript_pos', 'transcript_ref', 'transcript_alt', 'aa_pos', 'aa_ref', 'aa_alt', 'hgvs_c', 
                'hgvs_p', 'consequence', 'simplified_consequence', 'auth_reported_score',
                'auth_reported_func_class', 'splice_measure', 'gnomad_MAF', 'clinvar_sig_2025', 'clinvar_star_2025', 
                'clinvar_date_last_reviewed_2025', 'clinvar_sig_2018', 'clinvar_star_2018', 'clinvar_date_last_reviewed_2018', 
                'nucleotide_or_aa', 'Ensembl Transcript ID', 'RefSeq Transcript ID', 'Interval 1 Name', 'Interval 1 Range',
                'Interval 1 Class', 'Interval 2 Name', 'Interval 2 Range', 'Interval 2 Class', 'Interval 3 Name', 'Interval 3 Range', 
                'Interval 3 Class', 'Interval 4 Name', 'Interval 4 Range', 'Interval 4 Class', 'Interval 5 Name', 'Interval 5 Range', 
                'Interval 5 Class', 'Interval 6 Name', 'Interval 6 Range', 'Interval 6 Class', 'Flag', 'REVEL', 'REVEL_train', 
                'AM_score', 'AM_class', 'MutPred2', 'MP2_train', 'spliceAI_DS_AG', 'spliceAI_DS_AL', 'spliceAI_DS_DG', 'spliceAI_DS_DL',
                'spliceAI_DP_AG', 'spliceAI_DP_AL', 'spliceAI_DP_DG', 'spliceAI_DP_DL', 'ClinVar Variation Id_ClinGen_repo', 
                'Allele Registry Id_ClinGen_repo', 'Disease_ClinGen_repo', 'Mondo Id_ClinGen_repo', 'Mode of Inheritance_ClinGen_repo', 
                'Assertion_ClinGen_repo', 'Applied Evidence Codes (Met)_ClinGen_repo', 'Applied Evidence Codes (Not Met)_ClinGen_repo', 
                'Summary of interpretation_ClinGen_repo', 'PubMed Articles_ClinGen_repo', 'Expert Panel_ClinGen_repo', 
                'Guideline_ClinGen_repo', 'Approval Date_ClinGen_repo', 'Published Date_ClinGen_repo', 'Retracted_ClinGen_repo', 
                'Evidence Repo Link_ClinGen_repo', 'Uuid_ClinGen_repo', 'Updated_Classification_ClinGen_repo', 
                'Updated_Evidence Codes_ClinGen_repo','clinvar_sig_18_25','clnsig_group_18_25','StandardizedClass','ExC_points_2025', 
                'ExC_points_2018', 'OddsNormal', 'OddsAbnormal', 'OP_points', 'Fxn_points', 'Points_REVEL_GeneSpecific_GenomeWide', 
                'Points_AM_GeneSpecific_GenomeWide', 'Points_MP2_GeneSpecific_GenomeWide', 'Total_Points_REVEL',
                'Total_Points_AM', 'Total_Points_MP2', 'Class_REVEL', 
                'Class_AM', 'Class_MP2', 'Conflicting_REVEL', 'Conflicting_AM',
                'Conflicting_MP2']

# Apply column renames to all dataframes in dictionary. Column selection
# happens separately below, since the two output files (representative-only
# and with-secondary-variants) select a different column set.
for key in dfs_final:
    dfs_final[key] = dfs_final[key].rename(columns=rename_dict)

In [ ]:
# Two versions of Supplementary Data 5:
# - dfs_final_representative_only: the classic shape -- exactly one row per
#   variant, no Variant_Role column. Identical to what this pipeline always
#   produced, except that ties between multiple candidate NT variants for a
#   shared amino-acid substitution are now broken deterministically (see
#   docs/variant_classification.md's aa-stage tie-break section) instead of
#   by pandas' incidental row order.
# - dfs_final_with_secondary_variants: every candidate row, including the
#   ones that lost that tie-break (Variant_Role="secondary"), for anyone who
#   wants to see every NT variant considered for a shared substitution
#   rather than just the chosen representative.
dfs_final_representative_only = {
    key: df[df["Variant_Role"] == "primary"][column_order] for key, df in dfs_final.items()
}
dfs_final_with_secondary_variants = {
    key: df[column_order + ["Variant_Role"]] for key, df in dfs_final.items()
}


In [ ]:
out_folder = PREDICTOR_CALIBRATION_GENE_SPECIFIC_DIR
for name, df in dfs_final_representative_only.items():
    df.to_csv(out_folder / f"{name}.csv", index=False)

out_folder_with_secondary_variants = PREDICTOR_CALIBRATION_GENE_SPECIFIC_DIR.parent / "gene_specific_with_secondary_variants"
out_folder_with_secondary_variants.mkdir(parents=True, exist_ok=True)
for name, df in dfs_final_with_secondary_variants.items():
    df.to_csv(out_folder_with_secondary_variants / f"{name}.csv", index=False)


In [ ]:
import pandas as pd
import gzip
import shutil
from pathlib import Path


def build_supplementary_data_5(folder, output):
    with pd.ExcelWriter(output) as writer:
        for csv_file in folder.glob("*.csv"):
            df = pd.read_csv(csv_file)
            sheet_name = csv_file.stem[:31]
            df.to_excel(writer, sheet_name=sheet_name, index=False)
    with open(output, 'rb') as f_in:
        with gzip.open(str(output) + '.gz', 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)

In [ ]:
# Classic shape: one row per variant, no Variant_Role column -- identical to
# what this pipeline always produced (see the column_order cell above).
# Split into its own cell (and split from the with-secondary-variants call
# below) so each ExcelWriter run gets its own ExecutePreprocessor timeout
# budget -- writing ~480k rows x 107 cols through openpyxl takes a few
# minutes on its own, and the two calls combined were exceeding a single
# 600s cell timeout.
build_supplementary_data_5(
    PREDICTOR_CALIBRATION_GENE_SPECIFIC_DIR,
    SUPPLEMENTARY_DATA_DIR / "Supplementary_Data_5.xlsx",
)

In [ ]:
# Every candidate row, including secondary (non-representative) NT variants.
build_supplementary_data_5(
    PREDICTOR_CALIBRATION_GENE_SPECIFIC_DIR.parent / "gene_specific_with_secondary_variants",
    SUPPLEMENTARY_DATA_DIR / "Supplementary_Data_5.with_secondary_variants.xlsx",
)